# 04 — Rank feature lab

Sandbox for trying a new fact as a Measure. Scoring is the production
algorithm: each weighted Measure calibrated against random DayHike pairs, then
the weighted mean (`rank_dayhikes` / `score_rows`, ADR-0013). **Do not train a
ranker here.**

Production ranking itself is walked through in [`03_ranking.ipynb`](03_ranking.ipynb).
Load GeometryStore as GeoPandas in [`04a_geometry.ipynb`](04a_geometry.ipynb).
Geometry and raw amenity points live in GeometryStore
(`{catalog}.geometry.sqlite`), not in Catalog. `build-similarity` turns them
into the facts in `{catalog}.similarity/`.

## How to use this as a sandbox

1. Pick a Seed (default: "Aljažev dom - Kredarica - Triglav" when the Catalog
   has it, else the first OSM-named DayHike) and the Measure weights (default:
   `DEFAULT_WEIGHTS`, the core Measures at 100).
2. Look at which facts actually vary.
3. Add a fact array in **Experimental facts** (GeometryStore, DEM, anything).
4. Run `rank_with_extra` and read the comparison table.
5. Ablate a Measure by giving it weight 0.

Nothing here writes back to the Catalog. Promoting a fact means changing
`FACT_COLUMNS` / `MEASURES` in `dayhike.similarity`.

Open `data/catalog.sqlite` plus `data/catalog.geometry.sqlite` when both exist
and `dayhike build-similarity` has run. Otherwise build the committed fixture.

DEM / SolarAspect: `dayhike fetch-dem` writes FABDEM GeoTIFFs into `data/dem/`
(gitignored). `dayhike sample-dem` copies those heights onto GeometryStore
vertices and Catalog elevation; `build-similarity` reads them for the
`southness` fact. Without a covering file this notebook prints `dem skipped`
and still ranks.


In [ ]:
import math
import os
import sqlite3
import tempfile
from pathlib import Path

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

import numpy as np

from dayhike import (
    Catalog,
    build_catalog,
    build_similarity,
    geometry_store_path_for,
    rank_dayhikes,
    similarity_dir_for,
)
from dayhike.catalog import EARTH_RADIUS_M
from dayhike.similarity import (
    CLIMATE_MONTH_COLUMNS,
    DEFAULT_WEIGHTS,
    FACT_COLUMNS,
    MEASURES,
    calibrate,
    measure_scores,
    weighted_mean,
)

local = repo / "data" / "catalog.sqlite"
local_geometry = geometry_store_path_for(local)
if (
    local.is_file()
    and local_geometry.is_file()
    and similarity_dir_for(local).is_dir()
    and "PYTEST_CURRENT_TEST" not in os.environ
):
    catalog_path = local
    geometry_path = local_geometry
    catalog = Catalog.open(catalog_path)
else:
    catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    catalog = build_catalog(repo / "tests" / "fixtures" / "catalog.osm", catalog_path)
    build_similarity(catalog_path, catalog_path.parent / "dem")
    geometry_path = geometry_store_path_for(catalog_path)

store = catalog.similarity
geo = sqlite3.connect(f"file:{geometry_path}?mode=ro", uri=True)  # never written here
print("catalog", catalog_path)
print("geometry_store", geometry_path)
print("similarity", similarity_dir_for(catalog_path))
print(catalog.ingest_report)
print("name_origin")
for origin, n in geo.execute(
    "SELECT name_origin, COUNT(*) FROM dayhikes GROUP BY 1 ORDER BY 1"
):
    print(f"  {origin}\t{n}")


## Production baseline

`rank_dayhikes` is the control, with the production default weights (the core
Measures at 100, ADR-0016). Filter `name_origin != 'synthetic'` in
postprocessing if you only want OSM names. Change `SEED_NAME` or `WEIGHTS`
below to try another hike or another notion of "alike" (Measure name to 0-100;
a Measure left out is excluded).


In [ ]:
SEED_NAME = next(
    (hike.name for hike in catalog.dayhikes if hike.name == "Aljažev dom - Kredarica - Triglav"),
    next(hike.name for hike in catalog.dayhikes if hike.name_origin == "osm_name"),
)
WEIGHTS = {name: weight for name, weight in DEFAULT_WEIGHTS.items() if weight > 0}
print("seed", SEED_NAME)
production = rank_dayhikes(catalog, [SEED_NAME], WEIGHTS)
for item in production.items[:20]:
    print(
        f"prod_rank\t{item.dayhike.id}\t{item.dayhike.name}\t"
        f"{item.dayhike.name_origin}\t{item.score:.6f}"
    )

named = [hike for hike in catalog.dayhikes if hike.name_origin == "osm_name"]
print("osm_name_only", len(named), "of", len(catalog.dayhikes))


## What the facts do

Missingness and spread tell you which facts can move a ranking. A fact that is
all-NaN or constant cannot separate neighbours.


In [ ]:
import os

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")

import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)
import pandas as pd

ids = [hike.id for hike in catalog.dayhikes]
names = [hike.name for hike in catalog.dayhikes]
by_id = {hike.id: hike for hike in catalog.dayhikes}
fact_frame = pd.DataFrame(np.asarray(store.facts).T, columns=FACT_COLUMNS, index=ids)
summary_frame = fact_frame.drop(columns=list(CLIMATE_MONTH_COLUMNS))  # the QueryMonth copies
summary = pd.DataFrame(
    {
        "missing": summary_frame.isna().mean(),
        "std": summary_frame.std(),
        "min": summary_frame.min(),
        "max": summary_frame.max(),
    }
)
print(summary.to_string())

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
plot_cols = [
    "length_m",
    "elevation_gain_m",
    "difficulty_rank",
    "water_count",
    "dry_stretch_m",
    "climb_grade",
]
for ax, col in zip(axes.ravel(), plot_cols):
    values = fact_frame[col].dropna()
    if values.empty:
        ax.set_title(f"{col} (all missing)")
        continue
    ax.hist(values, bins=min(20, max(5, values.nunique())), color="#0f766e", edgecolor="white")
    ax.set_title(col)
fig.tight_layout()
show_fig(fig)


## Sandbox helpers

`rank_with_extra` is the whole experimental loop: score the Seed against every
DayHike on each weighted Measure, calibrate, add experimental facts as `ratio`
Measures calibrated the same way (against random pairs drawn here), and take
the weighted mean. `compare_to_production` prints rank deltas against the same
loop without the extras. Unlike `rank_dayhikes`, the loop keeps every Twin,
so a duplicate relation of the Seed's neighbour can sit next to it. Edit
**Experimental facts** in the next code cell.


In [ ]:
rng = np.random.default_rng(0)
left, right = rng.integers(0, len(ids), size=(2, 20_000))
left, right = left[left != right], right[left != right]
seed_row = ids.index(next(hike.id for hike in catalog.dayhikes if hike.name == SEED_NAME))
rows = np.array([row for row in range(len(ids)) if row != seed_row])


def ratio(a, b, floor):
    a, b = np.maximum(a, floor), np.maximum(b, floor)
    return np.minimum(a, b) / np.maximum(a, b)


def mid_rank(null, scores):
    null = np.sort(null[~np.isnan(null)])
    lo, hi = np.searchsorted(null, scores, "left"), np.searchsorted(null, scores, "right")
    return np.where(np.isnan(scores), np.nan, (lo + hi) / (2 * max(len(null), 1)))


def rank_with_extra(weights, extra):
    """(id, name, similarity) best first; ``extra`` maps a name to (facts, floor)."""
    columns, w = [], []
    for measure in MEASURES:
        if weights.get(measure.name, 0) > 0:
            raw = measure_scores(store, measure, seed_row, rows)
            columns.append(calibrate(store, measure.name, raw))
            w.append(weights[measure.name])
    for name, (values, floor) in extra.items():
        if weights.get(name, 0) > 0:
            null = ratio(values[left], values[right], floor)
            columns.append(mid_rank(null, ratio(values[seed_row], values[rows], floor)))
            w.append(weights[name])
    similarity = weighted_mean(np.column_stack(columns), np.array(w, dtype=float))
    order = [i for i in np.argsort(-similarity, kind="stable") if not np.isnan(similarity[i])]
    return [(ids[rows[i]], names[rows[i]], float(similarity[i])) for i in order]


baseline = rank_with_extra(WEIGHTS, {})


def compare_to_production(experimental, label="experiment"):
    prod_rank = {dayhike_id: i for i, (dayhike_id, _, _) in enumerate(baseline, start=1)}
    prod_score = {dayhike_id: score for dayhike_id, _, score in baseline}
    rows_out = []
    for rank, (dayhike_id, name, score) in enumerate(experimental, start=1):
        rows_out.append(
            {
                "id": dayhike_id,
                "name": name,
                "prod_rank": prod_rank.get(dayhike_id),
                f"{label}_rank": rank,
                "delta_rank": (prod_rank.get(dayhike_id) or 0) - rank,
                "prod_similarity": prod_score.get(dayhike_id),
                f"{label}_similarity": score,
            }
        )
    table = pd.DataFrame(rows_out)
    print(table.head(15).to_string(index=False))
    order_changed = [row[0] for row in experimental] != [row[0] for row in baseline]
    print("order_changed", order_changed)
    return table


## Experimental facts

Default extras from GeometryStore:

- `vertex_count` — how many points the assembled geometry has.
- `sinuosity` — route length / start–end great-circle (1 = straight).

Add your own arrays of `len(ids)` to `extra` as `(values, floor)` and give them
a weight in `exp_weights`.


In [ ]:
# One pass over the vertices primary key: count, first and last vertex per DayHike.
ends_rows = geo.execute(
    '''
    SELECT m.dayhike_id, m.n, a.latitude, a.longitude, b.latitude, b.longitude
    FROM (SELECT dayhike_id, COUNT(*) AS n, MAX(seq) AS last FROM vertices GROUP BY 1) m
    JOIN vertices a ON a.dayhike_id = m.dayhike_id AND a.seq = 0
    JOIN vertices b ON b.dayhike_id = m.dayhike_id AND b.seq = m.last
    '''
).fetchall()
vertex_count_by_id = {row[0]: row[1] for row in ends_rows}
vertex_count = np.array(
    [float(vertex_count_by_id.get(hike_id, 0)) for hike_id in ids], dtype=float
)
first_last = {row[0]: row[2:] for row in ends_rows}


def haversine_m(lat1, lon1, lat2, lon2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * EARTH_RADIUS_M * math.asin(math.sqrt(a))


sinuosity = np.ones(len(ids), dtype=float)
for index, hike_id in enumerate(ids):
    ends = first_last.get(hike_id)
    if not ends:
        continue
    chord = haversine_m(*ends)
    if chord > 0:
        sinuosity[index] = by_id[hike_id].length_m / chord

# --- edit here ---
extra = {
    "vertex_count": (vertex_count, 2.0),
    "sinuosity": (sinuosity, 1.0),
}
exp_weights = {**WEIGHTS, "vertex_count": 50, "sinuosity": 100}
exp_ranking = rank_with_extra(exp_weights, extra)
for dayhike_id, name, score in exp_ranking[:20]:
    print(f"exp_rank\t{dayhike_id}\t{name}\t{score:.6f}")
compare_to_production(exp_ranking)


## Ablation

Give some Measures weight 0 (here: the altitude ones) and see whether order
moves. That answers "does this Measure earn its keep for this Seed?".


In [ ]:
ALTITUDE = ("highest", "lowest", "elevation_gain")
ablate_ranking = rank_with_extra({**WEIGHTS, **dict.fromkeys(ALTITUDE, 0)}, {})
print("dropped", list(ALTITUDE))
compare_to_production(ablate_ranking, label="no_altitude")


## Map the Seed and its neighbours

GeometryStore vertices for the Seed plus the top production neighbours, one
panel each (on a real Catalog the neighbours can be anywhere in Europe).
Amenities are the raw points (including Viewpoint / Peak / Shelter) near each
line.


In [ ]:
from shapely.geometry import LineString, Point
import geopandas

seed_hike = by_id[ids[seed_row]]
show_ids = {seed_hike.id, *[item.dayhike.id for item in production.items[:8]]}
vertex_rows = list(
    geo.execute(
        "SELECT dayhike_id, seq, latitude, longitude FROM vertices "
        f"WHERE dayhike_id IN ({','.join('?' * len(show_ids))}) ORDER BY dayhike_id, seq",
        tuple(show_ids),
    )
)
by_hike = {}
for dayhike_id, _seq, lat, lon in vertex_rows:
    by_hike.setdefault(dayhike_id, []).append((lon, lat))
lines = [
    {
        "id": dayhike_id,
        "name": by_id[dayhike_id].name,
        "is_seed": dayhike_id == seed_hike.id,
        "geometry": LineString(coords),
    }
    for dayhike_id, coords in by_hike.items()
    if len(coords) >= 2
]
gdf = geopandas.GeoDataFrame(lines, crs="EPSG:4326").set_index("id")
# Seed first, then the neighbours in production order; each panel has its own
# extent, so neighbours far apart in Europe stay readable.
order = [hike_id for hike_id in [seed_hike.id, *[item.dayhike.id for item in production.items[:8]]] if hike_id in gdf.index]
PAD = 0.01  # degrees around each line for its amenity points
boxes = [gdf.loc[hike_id].geometry.bounds for hike_id in order]
amenity_rows = []
if boxes:
    where = " OR ".join("(latitude BETWEEN ? AND ? AND longitude BETWEEN ? AND ?)" for _ in boxes)
    params = [v for w, s_, e, n in boxes for v in (s_ - PAD, n + PAD, w - PAD, e + PAD)]
    amenity_rows = geo.execute(f"SELECT kind, latitude, longitude FROM amenities WHERE {where}", params).fetchall()
amenity_gdf = geopandas.GeoDataFrame(
    [{"kind": kind, "geometry": Point(lon, lat)} for kind, lat, lon in amenity_rows],
    columns=["kind", "geometry"],
    crs="EPSG:4326",
)
kinds = sorted(set(amenity_gdf["kind"]))
colors = dict(zip(kinds, plt.cm.tab10.colors))
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax in axes.ravel():
    ax.set_axis_off()
for ax, hike_id, (w, s_, e, n) in zip(axes.ravel(), order, boxes):
    line = gdf.loc[[hike_id]]
    line.plot(ax=ax, color="#c2410c" if hike_id == seed_hike.id else "#1d4ed8", linewidth=2.5)
    ax.scatter(*line.geometry.iloc[0].coords[0], color="black", s=40, zorder=5)
    near = amenity_gdf.cx[w - PAD : e + PAD, s_ - PAD : n + PAD]
    for kind, points in near.groupby("kind"):
        points.plot(ax=ax, color=colors[kind], markersize=12, label=kind)
    label = "Seed" if hike_id == seed_hike.id else f"#{order.index(hike_id)}"
    ax.set_title(f"{label}: {by_id[hike_id].name[:40]}", fontsize=8)
handles = [plt.Line2D([], [], marker="o", linestyle="", color=colors[k], label=k) for k in kinds]
handles.append(plt.Line2D([], [], marker="o", linestyle="", color="black", label="Trailhead"))
if handles:
    fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 6), fontsize=8)
fig.suptitle(f"Seed {SEED_NAME!r} and its production neighbours, with raw amenity points")
fig.tight_layout(rect=(0, 0.04, 1, 0.97))
show_fig(fig)
print("mapped_dayhikes", len(order), "amenities", len(amenity_gdf))


## Elevation profile (GeometryStore `vertices.ele`)

OSM `ele` on the fixture Ridge Walk; FABDEM metres after `sample-dem` on a
real Catalog. This is not SolarAspect — it is the along-track height the
Catalog elevation stats come from.


In [ ]:
profile = list(
    geo.execute(
        "SELECT seq, latitude, longitude, ele FROM vertices WHERE dayhike_id = ? ORDER BY seq",
        (seed_hike.id,),
    )
)
seq = [row[0] for row in profile]
ele = [row[3] for row in profile]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(seq, ele, color="#0f766e", marker="o")
ax.set_xlabel("vertex seq")
ax.set_ylabel("ele (m)")
ax.set_title(f"Elevation profile: {SEED_NAME}")
fig.tight_layout()
show_fig(fig)
print("profile_points", len(profile), "with_ele", sum(z is not None for z in ele))


## DEM and SolarAspect

`dayhike fetch-dem` then `dayhike sample-dem` persist FABDEM heights onto
GeometryStore vertices and Catalog elevation min/max/gain. `build-similarity`
reads FABDEM again for the SolarAspect fact `southness` (`1` = south-facing,
`0` = north-facing), which the `southness` Measure compares. Without a covering
GeoTIFF this cell prints `dem skipped` and `southness` is missing.


In [ ]:
from dayhike.dem import DEFAULT_DEM_DIR
from dayhike.similarity import dem_heights

points = np.array(
    list(geo.execute("SELECT latitude, longitude FROM vertices LIMIT 20")), dtype=float
)
samples = dem_heights(repo / DEFAULT_DEM_DIR, points[:, 0], points[:, 1])
finite = samples[np.isfinite(samples)]
if not finite.size:
    print("dem skipped: no covering FABDEM tile in data/dem/")
else:
    print("dem sampled", "n", finite.size, "min", finite.min(), "max", finite.max())
    print("solar aspect is DEM-derived; not live weather")
southness = fact_frame["southness"]
print("southness", "finite", int(southness.notna().sum()), "of", len(southness))
geo.close()


## Promoting a fact

When an experimental fact earns its keep:

1. Compute it in `dayhike.similarity` (`FACT_COLUMNS` and the facts build) so
   `build-similarity` stores it for every DayHike.
2. Add a `Measure` for it to `MEASURES` (kind and constant).
3. Rerun `dayhike build-similarity`. A new or changed fact rebuilds the facts; a
   Measure change alone needs only `--calibrate-only`.
4. Keep DriveTime out of the facts — it stays a query-time filter.

This notebook does not do those writes. Re-run `rank_dayhikes` after the
package change; that becomes the new baseline cell at the top.
